In [1]:
## Tokenization Example
import tiktoken
# Input text
text ="Generative AI is transforming industries"

# Load tokenizer
encoding = tiktoken.get_encoding("cl100k_base")

# Convert text into token IDs
token_ids = encoding.encode(text)

# Convert token IDs back to tokens
tokens = [encoding.decode([token]) for token in token_ids]
print("Original Text:")
print(text)
print("\nTokens:")
print(tokens)
print("\nToken IDs:")
print(token_ids)
print("\nTotal Tokens:")
print(len(token_ids))

Original Text:
Generative AI is transforming industries

Tokens:
['Gener', 'ative', ' AI', ' is', ' transforming', ' industries']

Token IDs:
[5648, 1413, 15592, 374, 46890, 19647]

Total Tokens:
6


In [2]:
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

# Input Sentences
sentences = [
    "Artificial Intelligence is transforming industries.",
    "AI is reshaping the modern business landscape.",
    "Cooking pasta requires boiling water."
]

# Step 1: Tokenization
tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

tokens = tokenizer.tokenize(sentences[0])

print("Sentence:")
print(sentences[0])

print("\nTokens:")
print(tokens)

# Step 2: Embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(sentences)
print("\nEmbedding Shape:")
print(embeddings.shape)

print("\nFirst 10 Values of Sentence 1 Embedding:")
print(embeddings[0][:10])


c:\Users\User\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sentence:
Artificial Intelligence is transforming industries.

Tokens:
['artificial', 'intelligence', 'is', 'transforming', 'industries', '.']

Embedding Shape:
(3, 384)

First 10 Values of Sentence 1 Embedding:
[ 0.03608525 -0.02447888  0.05895456 -0.00321432  0.04381187  0.01859978
 -0.01787441 -0.00463145 -0.01620272 -0.03146704]


In [5]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables
load_dotenv(override=True)

# Create Groq client
client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# Generate response
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "Hello, can you provide a brief overview of the latest advancements in generative AI?"
        }
    ],
    temperature=0.1,
    max_tokens=100
)

print(response.choices[0].message.content)

In [7]:
import os
from dotenv import load_dotenv
from openai import OpenAI


# Load API key
load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# Sample document
text = """
Refund Policy:
Customers can request refund within 7 days of purchase.
Refunds are not applicable for digital products after download.
Processing time is 5-10 business days.
"""


# Step 1: Chunking
def chunk_text(text, chunk_size=50):
    words = text.split()
    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size)
    ]


chunks = chunk_text(text)


# Step 2: Generate embeddings
index = []


for i, chunk in enumerate(chunks):

    response = client.embeddings.create(
        model="openai/text-embedding-3-small",
        input=chunk
    )

    embedding = response.data[0].embedding

    index.append({
        "id": i,
        "content": chunk,
        "embedding": embedding
    })


# Print indexed chunks
for item in index:
    print(f"\nChunk {item['id']}:")
    print(item['content'])
    print(f"Embedding Dimension: {len(item['embedding'])}")


Chunk 0:
Refund Policy: Customers can request refund within 7 days of purchase. Refunds are not applicable for digital products after download. Processing time is 5-10 business days.
Embedding Dimension: 1536


In [ ]:
#Retrival code
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)


# Document chunks
texts = [
    "Customers can get a refund within 7 days.",
    "Refund requests must be submitted online.",
    "Customers can return damaged products within 14 days."
]


# Create embeddings for the chunks
response = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=texts
)


# Store text + embedding
index = []

for text, item in zip(texts, response.data):
    index.append({
        "text": text,
        "embedding": np.array(item.embedding)
    })


# User query
query = "What is the refund policy?"


# Create query embedding
response = client.embeddings.create(
    model="openai/text-embedding-3-small",
    input=query
)

query_vector = np.array(response.data[0].embedding)


# Compare query with every document chunk
scores = []

for item in index:
    vector = item["embedding"]

    similarity = np.dot(vector, query_vector) / (
        np.linalg.norm(vector) *
        np.linalg.norm(query_vector)
    )

    scores.append(similarity)


# Find the most similar chunk
best = np.argmax(scores)


print("Query:", query)
print("Best Chunk:", index[best]["text"])
print("Similarity:", scores[best])

Query: What is the refund policy?
Best Chunk: Customers can get a refund within 7 days.
Similarity: 0.5525496023917177


In [ ]:
# Entire RAG code using LlamaIndex + Local Sentence Transformers Embeddings + Groq

import os
from dotenv import load_dotenv

from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding


# Load environment variables

load_dotenv(override=True)



# Local embedding configuration

Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


# Sample document

text = """
LlamaIndex is a framework for building applications using
Large Language Models and external data.

It supports Retrieval Augmented Generation (RAG).
The main steps in a RAG pipeline are data ingestion,
chunking, indexing, retrieval, and response generation.
"""


# Create document

document = Document(text=text)


# Chunking
nodes = Settings.node_parser.get_nodes_from_documents(
    [document]
)

print("Number of chunks:", len(nodes))


# Show chunks

for i, node in enumerate(nodes):
    print(f"\nChunk {i + 1}:")
    print(node.get_content())


# Indexing

index = VectorStoreIndex(
    nodes
)

print("\nIndex created")


# Create retriever

retriever = index.as_retriever(
    similarity_top_k=2
)


# User query

user_query = "What is LlamaIndex ?"

print("\nUser Query:")
print(user_query)


# Retrieve relevant chunks

retrieved_response = retriever.retrieve(
    user_query
)



print("\nRetrieved Chunks:")

# Display retrieved chunks

for i, item in enumerate(retrieved_response):

    chunk = item.node.get_content()

    print(f"\nChunk {i + 1}:")
    print(chunk)

    print("Similarity Score:")
    print(item.score)

    


Number of chunks: 1

Chunk 1:
LlamaIndex is a framework for building applications using
Large Language Models and external data.

It supports Retrieval Augmented Generation (RAG).
The main steps in a RAG pipeline are data ingestion,
chunking, indexing, retrieval, and response generation.

Index created

User Query:
What is LlamaIndex ?

Retrieved Chunks:

Chunk 1:
LlamaIndex is a framework for building applications using
Large Language Models and external data.

It supports Retrieval Augmented Generation (RAG).
The main steps in a RAG pipeline are data ingestion,
chunking, indexing, retrieval, and response generation.
Similarity Score:
0.6473458552364584


In [27]:
# Continue RAG using LangChain

from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

# Load API key

groq_api_key = os.getenv("GROQ_API_KEY")


# Groq client

groq_client = Groq(
    api_key=groq_api_key
)


# Take user query as input

user_query = input("\nEnter your question: ")

print("\nUser Query:")
print(user_query)


# Retrieve relevant chunks using existing LlamaIndex retriever

retrieved_nodes = retriever.retrieve(user_query)


# Create context

context = ""

for i, item in enumerate(retrieved_nodes):

    print(f"\nChunk {i + 1}:")
    print(item.node.get_content())

    context += item.node.get_content() + "\n"
 

# Create LangChain prompt

prompt = ChatPromptTemplate.from_template(
    """
    You are a friendly assistant. Answer the user's question using only the context provided below.

    Context:
    {context}

    Question:
    {question}

    Answer in a simple and friendly way.
    """
)


# Create Groq LLM using LangChain

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    groq_api_key=groq_api_key
)


# Create LangChain chain

chain = prompt | llm


# Generate answer

response = chain.invoke(
    {
        "context": context,
        "question": user_query
    }
)


# Display final response

print("\nResponse:")
print(response.content)


User Query:
Explain Llama index in 2 lines

Chunk 1:
LlamaIndex is a framework for building applications using
Large Language Models and external data.

It supports Retrieval Augmented Generation (RAG).
The main steps in a RAG pipeline are data ingestion,
chunking, indexing, retrieval, and response generation.

Response:
LlamaIndex lets you build apps that mix large language models with your own data.  
It does this by ingesting, chunking, indexing, retrieving, and then generating responses.
